# Hafta 6 — Örnek Tabanlı ve Olasılıksal Modeller

> Hücreleri **yukarıdan aşağıya sırayla** çalıştırın; sonraki hücreler önceki hücrelerde tanımlanan değişken ve fonksiyonları kullanır. Tek başına çalışmayan gösterim amaçlı kod parçaları markdown olarak bırakılmıştır. Veri setleri ilk çalıştırmada internetten indirilir.

In [ ]:
# Gerekli paketler kurulu değilse aşağıdaki satırın başındaki # işaretini kaldırıp çalıştırın
# %pip install numpy pandas matplotlib seaborn scikit-learn scipy

## Basit Anlatım: Bu Hafta Ne Öğreneceğiz?

**En yakın komşu yöntemi,** "Bana arkadaşını söyle, sana kim olduğunu söyleyeyim." sözüne dayanır. Yeni bir örneğin hangi gruba ait olduğuna karar vermek için ona en çok benzeyen birkaç örneğe bakılır. Bu örneklerin çoğunluğu hangi gruptaysa yeni örnek de o gruba atanır.

**Naive Bayes** ise ipuçlarını tartar. Bir e-postada "bedava", "kazandınız" ve "hemen tıklayın" ifadeleri geçiyor olsun. Yöntem, bu kelimelerin spam iletilerde ve normal iletilerde ne sıklıkla kullanıldığına bakar. Hangi tarafın ipuçları ağır basarsa e-posta o gruba dahil edilir.

---

## Bu Haftanın Hedefleri

- k-NN algoritmasını elle uygulamak ve sıfırdan kodlamak
- Mesafe ölçülerini ve boyutluluk lanetini anlamak
- Bayes teoremini yorumlamak
- Naive Bayes ile metin sınıflandırmak

---

## 1. k-En Yakın Komşu (k-NN)

### Çalışma mantığı

1. Yeni örneğin eğitim verisindeki **tüm örneklere** olan mesafesini hesapla.
2. En yakın **k** komşuyu bul.
3. Komşular arasında en çok görülen sınıfı seç.

k-NN eğitim sırasında hiçbir şey öğrenmez; sadece veriyi saklar. Bütün hesap tahmin anında yapılır.

### Elle örnek

Meyveler: x₁ = ağırlık, x₂ = tatlılık. Yeni meyve q = (3.5, 3).

| Meyve | Konum | q'ya mesafe | Sınıf |
|---|---|---|---|
| C | (4, 3) | 0.50 | Armut |
| E | (3, 2) | 1.12 | Elma |
| D | (5, 4) | 1.80 | Armut |
| B | (2, 1) | 2.50 | Elma |
| A | (1, 1) | 3.20 | Elma |

Mesafe Öklid formülüyle hesaplanır, örneğin C için √((4−3.5)² + (3−3)²) = 0.5.

| k | Komşular | Tahmin |
|---|---|---|
| 1 | C | Armut |
| 3 | C, E, D | Armut (2–1) |
| 5 | Hepsi | Elma (3–2) |

**k sonucu değiştirebilir.** k çok küçükse model gürültüye tepki verir (aşırı öğrenme), çok büyükse her şeye çoğunluk sınıfını söyler (eksik öğrenme). En iyi k çapraz doğrulamayla seçilir.

### Mesafe ölçüleri

p = (1, 2) ve q = (4, 6) için:

| Ölçü | Hesap | Sonuç |
|---|---|---|
| Öklid (kuş uçuşu) | √(3² + 4²) | 5 |
| Manhattan (şehir blokları) | 3 + 4 | 7 |
| Kosinüs | Vektörlerin yönüne bakar | Metin verisinde yaygın |

> Mesafe kullanan her yöntemde **ölçekleme şarttır**; aksi halde büyük sayılı öznitelik baskın çıkar.

### Sıfırdan k-NN

In [ ]:
import numpy as np
from collections import Counter

class BenimKNN:
    def __init__(self, k=3):
        self.k = k

    def fit(self, X, y):
        self.X, self.y = np.asarray(X, dtype=float), np.asarray(y)   # sadece sakla
        return self

    def predict(self, X):
        tahminler = []
        for x in np.asarray(X, dtype=float):
            mesafe = np.sqrt(((self.X - x) ** 2).sum(axis=1))
            komsular = self.y[np.argsort(mesafe)[:self.k]]
            tahminler.append(Counter(komsular).most_common(1)[0][0])
        return np.array(tahminler)

X = [[1, 1], [2, 1], [4, 3], [5, 4], [3, 2]]
y = ["Elma", "Elma", "Armut", "Armut", "Elma"]
for k in [1, 3, 5]:
    print(k, BenimKNN(k).fit(X, y).predict([[3.5, 3]]))

### Boyutluluk laneti

Boyut sayısı arttıkça en yakın ve en uzak komşu arasındaki fark kaybolur; "yakınlık" anlamını yitirir:

In [ ]:
rng = np.random.default_rng(0)
for d in [2, 10, 100, 1000]:
    veri, q = rng.random((500, d)), rng.random(d)
    m = np.sqrt(((veri - q) ** 2).sum(axis=1))
    print(f"{d:4d} boyut: en yakın / en uzak = {m.min() / m.max():.2f}")

Oran 1'e yaklaştıkça bütün noktalar birbirine eşit uzaklıkta görünür. Bu yüzden k-NN çok boyutlu verilerde zorlanır.

---

## 2. Bayes Teoremi ve Naive Bayes

### Bayes teoremi

```
P(sınıf | veri) = P(veri | sınıf) · P(sınıf) / P(veri)
```

Yeni bir kanıt gördüğümüzde önceki inancımızı günceller.

**Tıbbi test örneği:** Bir hastalık nüfusun %1'inde görülüyor. Test hastaların %99'unu yakalıyor ama sağlıklıların %5'ine de yanlışlıkla "pozitif" diyor. Testi pozitif çıkan biri gerçekten hasta mıdır?

10.000 kişi düşünelim:
- 100 hastadan 99'u pozitif çıkar.
- 9.900 sağlıklıdan 495'i yanlışlıkla pozitif çıkar.
- Toplam 594 pozitif sonucun yalnızca 99'u gerçek hastadır: 99 / 594 = **%16.7**

Hastalık nadir olduğu için yanlış alarmlar gerçek vakalardan sayıca fazladır. **Başlangıç olasılığı (önsel) sonucu güçlü biçimde etkiler.**

### Naive Bayes

Bir e-postanın kelimeleri (w₁, w₂, ...) verildiğinde hangi sınıfın daha olası olduğunu hesaplar. "Naif" varsayım şudur: Sınıf bilindiğinde kelimeler birbirinden bağımsızdır. Bu varsayım sayesinde hesap basit bir çarpıma dönüşür:

```
Skor(sınıf) = P(sınıf) × P(w₁ | sınıf) × P(w₂ | sınıf) × ...
```

### Elle örnek

| E-posta | Sınıf |
|---|---|
| bedava para kazan | spam |
| bedava hediye kazan | spam |
| para kazan | spam |
| toplantı yarın | normal |
| proje toplantı yarın | normal |
| para yarın | normal |

Spam iletilerde toplam 8 kelime, normal iletilerde 7 kelime var; sözlükte 7 farklı kelime bulunuyor.

**Laplace düzeltmesi:** "bedava" kelimesi normal iletilerde hiç geçmiyor. Bu yüzden olasılığı 0 çıkar ve bütün çarpımı sıfırlar. Bunu önlemek için her kelime sayısına 1 eklenir:

```
P(kelime | sınıf) = (kelime sayısı + 1) / (sınıftaki toplam kelime + sözlük boyutu)
```

**Yeni e-posta: "bedava para"**

```
spam:   0.5 × (2+1)/(8+7) × (2+1)/(8+7) = 0.5 × 0.200 × 0.200 = 0.0200
normal: 0.5 × (0+1)/(7+7) × (1+1)/(7+7) = 0.5 × 0.071 × 0.143 = 0.0051

P(spam) = 0.0200 / (0.0200 + 0.0051) ≈ %80  →  SPAM
```

Gerçek e-postalarda yüzlerce küçük olasılık çarpılınca sonuç bilgisayarda sıfıra yuvarlanır. Bu yüzden pratikte logaritmalar toplanır.

### Sıfırdan Naive Bayes

In [ ]:
class BenimNaiveBayes:
    def fit(self, belgeler, etiketler):
        self.siniflar = sorted(set(etiketler))
        sozluk = {k for b in belgeler for k in b.split()}
        self.log_prior, self.log_olas = {}, {}
        for s in self.siniflar:
            sinif_belge = [b for b, e in zip(belgeler, etiketler) if e == s]
            kelimeler = [k for b in sinif_belge for k in b.split()]
            self.log_prior[s] = np.log(len(sinif_belge) / len(belgeler))
            self.log_olas[s] = {k: np.log((kelimeler.count(k) + 1) / (len(kelimeler) + len(sozluk)))
                                for k in sozluk}
        return self

    def predict(self, belge):
        skor = {s: self.log_prior[s] + sum(self.log_olas[s].get(k, 0) for k in belge.split())
                for s in self.siniflar}
        return max(skor, key=skor.get)

belgeler = ["bedava para kazan", "bedava hediye kazan", "para kazan",
            "toplantı yarın", "proje toplantı yarın", "para yarın"]
etiketler = ["spam", "spam", "spam", "normal", "normal", "normal"]
nb = BenimNaiveBayes().fit(belgeler, etiketler)
print(nb.predict("bedava para"), nb.predict("yarın toplantı var mı"))   # spam normal

---

## 3. Laboratuvar: SMS Spam Tespiti

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

url = "https://raw.githubusercontent.com/justmarkham/pycon-2016-tutorial/master/data/sms.tsv"
df = pd.read_csv(url, sep="\t", header=None, names=["etiket", "mesaj"])
y = (df["etiket"] == "spam").astype(int)
X_tr, X_te, y_tr, y_te = train_test_split(df["mesaj"], y, test_size=0.2, random_state=42, stratify=y)

# Metin → kelime sayıları → Naive Bayes
model = make_pipeline(CountVectorizer(), MultinomialNB()).fit(X_tr, y_tr)
print(classification_report(y_te, model.predict(X_te), target_names=["ham", "spam"], digits=3))

# Model ne öğrendi? En güçlü spam kelimeleri
vec, nb_model = model[0], model[1]
oran = nb_model.feature_log_prob_[1] - nb_model.feature_log_prob_[0]
print("Spam göstergeleri:", vec.get_feature_names_out()[np.argsort(oran)[-10:]])

**Görevler:**
1. Naive Bayes ile `make_pipeline(TfidfVectorizer(), KNeighborsClassifier(5, metric="cosine"))` modelini 5-katlı çapraz doğrulamada F1 skoruyla karşılaştırın.
2. Kendi yazdığınız birkaç mesajı modele verip sonuçları yorumlayın.

---

## 4. Özet

| | k-NN | Naive Bayes |
|---|---|---|
| Fikir | En yakın komşuların oyu | Olasılıkları çarp, en büyüğü seç |
| Eğitim | Veriyi saklar | Kelime sayılarını hesaplar |
| Ölçekleme | Şart | Gerekmez |
| Çok boyutlu veri | Zorlanır | İyi çalışır |

## Kendinizi Sınayın

1. Bölüm 1'deki veride q = (2, 2) noktasını k = 3 ile sınıflandırın.
2. Hastalık nüfusun %10'unda görülseydi pozitif test sonucunun gerçek hastalık olasılığı ne olurdu?
3. Laplace düzeltmesi yapılmazsa ne sorun çıkar?

## Çalışma Alanı

Alıştırmalar ve "Kendinizi Sınayın" soruları için aşağıdaki hücreleri kullanın.